# Image Classification
Vincent Luong

## Introduction

Write a brief summary about image classification and pytorch

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import os
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import ToTensor
import torchvision.transforms as tt
from torchvision.utils import make_grid
from torch.utils.data.dataloader import DataLoader
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

## Importing the Dataset

We will need to first normalize the image tensors by subtracting the mean and dividing by the standard deviation across each channel.  As a result, the mean of the data across each channel is 0, and the standard deviation is 1 ~ N(0,1). Normallizing the data prevents the values from any one channel from disproportionately affecting the losses a nd gradients while training, simply by having a higher or wider range of values.

In [2]:
train = ImageFolder("../image-classification/data/seg_train", transform = tt.Compose([
    tt.Resize(64),
    tt.RandomCrop(64),
    tt.ToTensor(),
]))
train_dl = DataLoader(train, 64, shuffle=True, num_workers=3, pin_memory=True)

def get_mean_std(dl):
    sum_, squared_sum, batches = 0,0,0
    for data, _ in dl:
        sum_ += torch.mean(data, dim = ([0,2,3]))
        squared_sum += torch.mean(data**2, dim = ([0,2,3]))
        batches += 1
        
    mean = sum_/batches
    std = (squared_sum/batches - mean**2)**0.5
    return mean,std

mean, std = get_mean_std(train_dl)
mean, std

(tensor([0.4303, 0.4575, 0.4539]), tensor([0.2481, 0.2467, 0.2806]))

We will then apply random chosen transformations while loading images from the training set, more specifically, we will pad each image by 4 pixels, and then take a random crop of 64 x 64 pixels, and then flip the image horizontally with a 50% probability.  Since the transformation will be applied randomly and dynamically each time a particular image is loaded, the model sees a slightly different image in each *epoch* of training, allowing for a better generalization.

In [3]:
stats = ((0.4951, 0.4982, 0.4979), (0.2482, 0.2467, 0.2807))
train_transform = tt.Compose([
    tt.Resize(64),
    tt.RandomCrop(64),
    tt.RandomHorizontalFlip(),
    tt.ToTensor(),
    tt.Normalize(*stats,inplace=True)
])

test_transform = tt.Compose([
    tt.Resize(64),
    tt.RandomCrop(64),
    tt.ToTensor(),
    tt.Normalize(*stats,inplace=True)
])

## Creating a Train/Test Split

In [4]:
train = ImageFolder("../image-classification/data/seg_train", transform = train_transform)
test = ImageFolder("../image-classification/data/seg_test",transform = test_transform)

After importing the following images into our jupyter notebook, we will split our dataset into 2 parts.  In our case of **image classification**, we will have a training set `train_dat` for training our model; we will be working with this for model construction.  

Additionally, we also create a testing set `test_dat`, these images will be unseen and be used to test our model accuracy.

We will be using a 80/20 train/test split to train our model

In [7]:
torch.manual_seed(42)

test_size = int(len(train) * 0.2)
train_size = len(train) - test_size

train_dat, test_dat = random_split(train, [train_size, test_size])
len(train_dat), len(test_dat)

(11228, 2806)